# CP3-04 制造错误：失败超步仍然留下检查点

本 Notebook 是错误三部曲的第一步：`START → node_change_topic → (node_poem ‖ node_joke) → node_output → END`。写诗与写笑话位于同一超步；本节在 `node_joke` 中故意抛出异常，观察 LangGraph 如何中止当前调用并保存可恢复的失败现场。

## 运行前提

- Python 3.11+；已安装 `langgraph`、`langchain-deepseek`、`langgraph-checkpoint-postgres`、`psycopg`、`python-dotenv` 与 `loguru`。
- PostgreSQL 已启动，并通过环境变量 `LANGGRAPH_DB_URL` 提供连接串。
- 已配置 `DEEPSEEK_API_KEY`；模型名可通过 `DEEPSEEK_MODEL` 覆盖。

代码中的 `checkpointer.setup()` 是幂等的：首次运行负责建表，后续运行可以重复执行。代码捕获异常只是为了让 Notebook 单元正常结束；异常本身仍会发生，并且失败超步会写入 PostgreSQL。


In [ ]:
import os
import time
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, StateGraph
from loguru import logger

# 只从环境变量读取密钥和数据库地址，避免把凭据提交进 Notebook。
load_dotenv(override=True)
MODEL_NAME = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
DB_URL = os.getenv('LANGGRAPH_DB_URL')
if not os.getenv('DEEPSEEK_API_KEY'):
    raise RuntimeError('缺少 DEEPSEEK_API_KEY，请先配置 .env 或系统环境变量。')
if not DB_URL:
    raise RuntimeError('缺少 LANGGRAPH_DB_URL，请配置 PostgreSQL 连接串。')

model = ChatDeepSeek(
    model=MODEL_NAME,
    extra_body={'thinking': {'type': 'disabled'}},
)

# 图内部状态会随着超步逐步补齐，所以中间字段设为可选。
class OverAllState(TypedDict, total=False):
    topic: str
    poem: str
    joke: str
    final_output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    final_output: str

class IntentionalNodeError(RuntimeError):
    """CP3-04 专用的教学故障，避免掩盖真实配置错误。"""

# 这个模块使用全局索引轮换子主题；它只对新起运行生效，恢复时不会重跑。
topics = ['布偶猫', '狸花猫', '金渐层']
topic_index = 0

def node_change_topic(state: InputState) -> OverAllState:
    global topic_index
    logger.info('topic_index: {}', topic_index)
    sub_topic = topics[topic_index]
    topic_index = (topic_index + 1) % len(topics)
    return {'topic': f'{state["topic"]}:{sub_topic}'}

# 该节点与 node_joke 从同一父节点出发，因此在同一超步并行。
def node_poem(state: OverAllState) -> OverAllState:
    logger.info('node_poem 正在执行')
    topic = state['topic']
    response = model.invoke([HumanMessage(content=f'写一首关于{topic}主题的七言绝句')])
    return {'poem': response.content}

# 故意制造错误：失败任务没有写出 joke，但同超步中已经完成的 poem 可能已经写入检查点。
def node_joke(state: OverAllState) -> OverAllState:
    logger.info('node_joke 正在执行')
    time.sleep(5)
    raise IntentionalNodeError('人为抛异常：用于演示可恢复的失败节点')

def node_output(state: OverAllState) -> OutputState:
    logger.info('node_output 正在执行')
    final_output = (
        f'关于{state["topic"]}的七言绝句:{state["poem"]}'
        + chr(10)
        + f'笑话:{state["joke"]}'
    )
    return {'final_output': final_output}

# 构图：改主题 → 扇出两个并行节点 → 两个节点都完成后再汇总。
builder = StateGraph(
    state_schema=OverAllState,
    input_schema=InputState,
    output_schema=OutputState,
)
builder.add_node('node_change_topic', node_change_topic)
builder.add_node('node_poem', node_poem)
builder.add_node('node_joke', node_joke)
builder.add_node('node_output', node_output)
builder.add_edge(START, 'node_change_topic')
builder.add_edge('node_change_topic', 'node_poem')
builder.add_edge('node_change_topic', 'node_joke')
builder.add_edge('node_poem', 'node_output')
builder.add_edge('node_joke', 'node_output')
builder.add_edge('node_output', END)

from langgraph.checkpoint.postgres import PostgresSaver
THREAD_ID = os.getenv('CP3_ERROR_THREAD_ID', 'chapter03-05')
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # setup() 初始化检查点表；它不会删除既有历史。
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)
    print(graph.get_graph().draw_mermaid())
    config = {'configurable': {'thread_id': THREAD_ID}}

    # 预期这里会进入 node_joke 的异常分支；捕获只为让后续 Notebook 能继续运行。
    try:
        result = graph.invoke({'topic': '猫'}, config=config)
        print(result)
    except IntentionalNodeError as exc:
        print({'expected_error': str(exc), 'thread_id': THREAD_ID})


## 预期观察

1. `node_poem` 与 `node_joke` 同属一个并行超步；调度顺序不保证固定。
2. `invoke` 不能返回最终输出，因为 `node_joke` 失败后 `node_output` 不会执行。
3. 失败前已成功写出的 `poem` 结果和 `node_joke` 的错误信息会进入 `StateSnapshot.tasks`。
4. CP3-05 将读取同一个 `thread_id` 的历史，CP3-06 将从这个失败位置恢复。
